# 🏛️ The FAANG Vision ML Master Playbook: From Zero to Hyper-Scale

This playbook synthesizes everything learned across our `docs/` architecture, MLOps integrations (DagsHub, Roboflow, MLflow), and Systems Design principles. 
It maps the evolution of our **Retail Analytics MVP** from a local script (Level 0) to a fully fault-tolerant, highly-scaled enterprise system (Level 5).

If you understand the transitions between these levels, you understand FAANG-level Machine Learning Systems Engineering.

---

## Level 0: The Local Sandbox (Beginner / Research Phase)

**Objective**: Prove the model works on a single machine.
**Our Implementation**: `01_yolo_baseline.ipynb`, `02_bytetrack_tracking.ipynb`

### Characteristics
- **Data**: Downloaded manually, stored in `./datasets/` (Unversioned).
- **Code**: Jupyter Notebooks, monolithic scripts.
- **Compute**: Single local GPU or CPU. Synchronous execution.
- **Tracking**: `print(loss)` or basic local MLflow.
- **DSA Focus**: Understanding the $O(N^2)$ complexity of convolutions, the $O(B^2)$ bottleneck of Non-Maximum Suppression (NMS), and Kalman Filter $O(N)$ state estimation for tracking.

### Why this fails in production:
- **No Reproducibility**: "It worked on my machine."
- **Data Starvation**: The CPU spends more time decoding JPEG frames than the GPU spends doing tensor math. The GPU sits idle 80% of the time.

## Level 1: Reproducible Repositories (Intermediate / Software Engineer)

**Objective**: Make the project reproducible and modular for a team.
**Our Implementation**: `docs/REPO_ARCHITECTURE.md`, `src/vision_ml/`

### Characteristics
- **Code**: SOLID principles. Interfaces (`BaseDetector`). Config-driven (YAML) instead of hardcoded hyperparameters.
- **Data (DagsHub/DVC)**: Introduced **Content-Addressable Storage**. The dataset is hashed via MD5, stored in S3, and symlinked locally. Code and Data are tied to a specific Git commit.
- **Tracking (MLflow)**: Hyperparameters and metrics are logged. The `Trainer` class handles checkpoints.
- **Testing**: Pytest unit tests for preprocessing and interfaces.

### Why this fails in production:
- **Static Data**: The model is trained once on a perfect dataset. When the store lighting changes (Data Drift), the model's accuracy drops silently to 0%.

## Level 2: The Automated Data Engine (Senior / ML Engineer)

**Objective**: Close the loop. The model must improve itself using real-world data.
**Our Implementation**: `05_roboflow_data_engine.ipynb`, `04_evidently_drift.ipynb`

### Characteristics
- **Active Learning**: The inference pipeline uses **entropy/confidence thresholding**. If the model is 40%-60% confident, it saves the frame.
- **Data Flywheel (Roboflow)**: Saved frames are pushed to an annotation API. Humans (or Auto-labelers) correct the boxes, creating `Dataset v2`.
- **Drift Detection (Evidently)**: We use **Reservoir Sampling** ($O(1)$ memory) to sample the infinite video stream. Nightly, we run $O(K \log K)$ Kolmogorov-Smirnov tests to compare today's data distribution vs the training data distribution.
- **Automation**: Apache Airflow DAGs detect drift $\to$ pull `Dataset v2` $\to$ trigger a retraining pipeline.

### Why this fails in production:
- **Inference Scaling**: We are still processing frames synchronously. 1 Camera = 30 FPS. 100 Cameras = 3000 FPS. A single GPU will crash with an Out-of-Memory (OOM) error or terrible latency jitter.

## Level 3: Scaled & Optimized Inference (Staff / Infrastructure Engineer)

**Objective**: Decouple compute, optimize hardware utilization, and scale horizontally.
**Our Implementation**: `docs/scaling.md`, System Diagrams in `01_yolo_baseline.ipynb`

### Characteristics
- **Decoupled I/O**: 
  - Ingestion pods (Go/Rust) decode H264 via hardware (NVDEC) and push raw tensors to shared memory.
  - Inference pods (Python/Triton) pull from shared memory. 
- **Dynamic Batching**: A server like **NVIDIA Triton** queues incoming requests up to `Batch=8` to maximize GPU Tensor Core utilization, turning a memory-bandwidth bound problem into a compute-bound problem.
- **Hardware Compilation**: Models are exported from PyTorch eager mode to **TensorRT**, fusing kernels (Conv+BatchNorm) and quantizing to INT8 for $3\times$ speedups.
- **Stateful Tracking Architecture**: YOLO is stateless (autoscales easily). ByteTrack is stateful. We use **Consistent Hashing** (e.g., hash the Camera ID) to route all frames from Camera 1 to Tracking Pod A, keeping the Kalman Filter state in local memory.

### Why this fails in production:
- **Long-Tail Edge Cases**: The model encounters a completely new object (e.g., a spilled liquid on the floor). It has never been trained on this. It fails silently. Humans are too slow to label thousands of these specific edge cases.

## Level 4: LLMOps & Foundation Model Oracles (Principal / AI Architect)

**Objective**: Solve the long-tail edge case problem automatically using Teacher-Student distillation.
**Our Implementation**: `07_llmops_vision_language_models.ipynb`

### Characteristics
- **Vision-Language Models (VLMs)**: We deploy an open-vocabulary Foundation Model like **Florence-2** or **PaliGemma** in the cloud.
- **LLMOps Architecture**: Serving a VLM requires **Continuous Batching** and **PagedAttention** (vLLM) to handle the massive KV-Cache state associated with Autoregressive decoding.
- **Knowledge Distillation (The Auto-Labeling Oracle)**:
  1. The fast, edge YOLO model detects an anomaly (or we use Vector Search / CLIP embeddings to find rare frames in our database).
  2. We send the frame to the VLM with the prompt: *"<OD> Find the spilled liquid."*
  3. The slow VLM accurately finds the bounding box (Zero-Shot).
  4. We convert the VLM output into YOLO labels (Pseudo-labeling).
  5. We fine-tune the fast YOLO model on this new data.
- **Result**: The fast model learns the slow model's knowledge, entirely autonomously.
- **Enterprise Pattern**: "Shadow Mode" deployment. The new student model runs alongside the old one in production, receiving real traffic but not making business decisions until its metrics statistically surpass the baseline (A/B testing / Canary).

### Why this fails in production:
- **Cluster Reliability**: We are now running distributed PyTorch training jobs (Ring-AllReduce) automatically, scaling YOLO inference, and hosting massive VLMs. If a network switch fails, or a GPU overheats, the entire automated pipeline hangs indefinitely.

## Level 5: Full Fault-Tolerant AIOps (The Enterprise Pinnacle)

**Objective**: The system monitors itself, heals itself, and survives hardware failure without human intervention.
**Our Implementation**: `08_aiops_automated_rca.ipynb`, `03_mlflow_pipeline.ipynb`

### Characteristics
- **Idempotent Pipelines**: Training jobs are designed to crash. When Airflow restarts a failed PyTorch `DistributedDataParallel` job, MLflow detects the `run_id`, loads the last epoch's checkpoint from S3, and resumes without corrupting metrics.
- **Infrastructure Anomaly Detection**: **Isolation Forests** continuously scan Prometheus metrics (GPU Temp, VRAM Usage). If a node starts thermal throttling, it is instantly cordoned.
- **Automated Root Cause Analysis (RCA)**: 
  - The system is modeled as a **Directed Acyclic Graph (DAG)**.
  - When a cascade of microservice failures occurs, a Reverse Topological Traversal algorithm identifies the *root* node (e.g., the Redis queue that OOM'd).
- **LLM SRE Agents**: An LLM agent is fed the root cause graph node and its recent Elasticsearch logs. It identifies the issue (*"Queue grew too fast due to ingestion spike"*) and automatically executes mitigation scripts (*"Scaling Inference replicas from 2 to 4"*).
- **Feature Store & Real-time Parity**: (The final FAANG pattern). Ensuring that the data transformations happening in offline training are *exactly identical* to the online streaming inference, enforced by a central Feature Store (like Feast or Hopsworks).

---

## 🏆 Summary: The FAANG Interview Cheat Sheet

If asked how to build a Retail Analytics Vision System:

1. **Don't start with the model.** Start with the **Data Engine** (Level 2). Explain how you capture edge cases using confidence thresholds and vector search (Roboflow).
2. **Explain the Latency constraints** (Level 3). Show you understand the Roofline model. Explain why batching is needed to saturate GPU compute, and why Tracker statefulness requires consistent routing.
3. **Solve the annotation bottleneck** (Level 4). Introduce VLMs as offline auto-labeling oracles to distill knowledge into fast edge models.
4. **Ensure reliability** (Level 5). Talk about idempotency, DAG-based root cause analysis, and how to survive a spot-instance cluster death mid-training.

This playbook bridges the gap between typing `model.fit()` and engineering planetary-scale AI systems.